# Create Frozen Train Validation Split

In [1]:
import pandas as pd
import numpy as np

Load data from preprocessing/normalization

In [2]:
DATA_DIR = "split_data"

contracts = pd.read_parquet(f"{DATA_DIR}/contracts.parquet")
documents = pd.read_parquet(f"{DATA_DIR}/documents.parquet")
categories = pd.read_parquet(f"{DATA_DIR}/categories.parquet")
annotation_sets = pd.read_parquet(f"{DATA_DIR}/annotation_sets.parquet")
spans = pd.read_parquet(f"{DATA_DIR}/spans.parquet")

# Some Context


As we split our data into training and validation, our goal is essentially choosing which context_group_ids go into train or validation. We use the 41 categories as the stratification signal, and then give every Contract and its related data the split of its context group.

For example, say contract1 has context_group_ids 'group a' and contract2 also has context_group_ids 'group a', then we wouldn't want contract1 to be split into train while contract2 is split into validation, contract1 and contract2 must be in the same split.

We use the 41 categories as the stratification signal to ensure that train and validation have similar distributions across the splits. Also, our 41 category presences is combined so that if any category appears in a context_group_id it is accounted for.



# Build a context-group × category matrix

We build a context-group x category matrix in order to track which categories are present in each context_group_ids


In [3]:
category_ids = categories["category_id"].drop_duplicates().tolist()

contract_groups = (
    documents[
        ["contract_id", "context_group_id"]
    ]
    .drop_duplicates()
)

contracts_with_groups = contracts.merge(
    contract_groups, on="contract_id", how="left"
)


#connect contracts to their categories
contract_category_groups = (
    annotation_sets[
        [
            "contract_id",
            "category_id",
            "is_impossible"
        ]
    ]
    .merge(
        contract_groups,
        on="contract_id",
        how="left"
    )
)

answerable_annotations = contract_category_groups[
    contract_category_groups["is_impossible"] == False
].copy()

group_category_presence = (
    answerable_annotations[
        [
            "context_group_id",
            "category_id"
        ]
    ]
    .drop_duplicates()
)

group_category_presence["present"] = 1

#matrix
group_category_matrix = (
    group_category_presence
    .pivot_table(
        index="context_group_id",
        columns="category_id",
        values="present",
        aggfunc="max",
        fill_value=0
    )
)

#categories as columns
group_category_matrix = (
    group_category_matrix
    .reindex(
        columns=category_ids,
        fill_value=0
    )
)



# Check/Assertations for matrix

In [4]:
# Exactly 41 category columns
assert group_category_matrix.shape[1] == 41

# Only 0 and 1 should occur
assert set(
    group_category_matrix.to_numpy().flatten()
).issubset({0, 1})

# Every context group should have exactly one row
assert (
    group_category_matrix.index.is_unique
)

# Begin Split

split units, groups

In [5]:
groups = group_category_matrix.index.to_numpy()

41 labels associated with the groups (binary)

In [6]:
Y = group_category_matrix.to_numpy()

we need to use seeded iterative multilabel stratification because we have 41 categories with groups that can have multiple categories. This makes it a multilabel problem. Iterative multilabel stratification tries to distribute those combinations across train and validation while respecting the 80/20 target. Our seed ensures that we have a reproducible split. The given seed is 20260901. It is our random state.

In [8]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=20260901
)

and then we split

In [9]:
train_idx, validation_idx = next(
    splitter.split(groups, Y)
)

next we track which groups are in train and validation

In [10]:
train_groups = groups[train_idx]
validation_groups = groups[validation_idx]

To freeze this split, we want to create a dataframe to document the split

In [11]:
split_map = pd.DataFrame({
    "context_group_id": groups,
    "split": np.where(
        np.isin(groups, train_groups),
        "train",
        "validation"
    )
})

In [17]:
split_map.head(15)

,context_group_id,split
0,context_0003419e35,train
1,context_003b6680f2,train
2,context_014a1c0939,train
3,context_04ebe8271f,train
4,context_05461c3198,train
5,context_06abafd8de,validation
6,context_06b999fb19,train
7,context_0723fe679b,train
8,context_0835bda997,train
9,context_089b029b55,validation


to check that the split is essentially 80/20

In [12]:
print(split_map["split"].value_counts())
print(
    split_map["split"].value_counts(normalize=True)
)

split
train         324
validation     83
Name: count, dtype: int64
split
train         0.796069
validation    0.203931
Name: proportion, dtype: float64


we also want to check that there is no overlap

In [13]:
assert set(train_groups).isdisjoint(
    set(validation_groups)
)

# Export Frozen Split

finally, we want to export the frozen train and validation split as a parquet for easy access for future use

In [19]:
import pyarrow

from pathlib import Path

output_dir = Path("frozen-split_data")
output_dir.mkdir(exist_ok=True)

split_map.to_parquet(output_dir / "frozen-split.parquet", index=False)